## Objectifs : 
Ce mini-projet vise à évaluer votre capacité à implémenter et analyser expérimentalement des méthodes liées à l’équité algorithmique, à l’interprétabilité et à la robustesse des modèles d’apprentissage automatique. 

L’accent est mis sur la qualité du protocole expérimental, la reproductibilité, et l’analyse critique des compromis observés entre performance, équité et robustesse.

In [1]:
import pandas as pd

df = pd.read_csv('../data/features.csv')
df.head(5)

,Attribute1,Attribute2,Attribute3,Attribute4,Attribute5,Attribute6,Attribute7,Attribute8,Attribute9,Attribute10,Attribute11,Attribute12,Attribute13,Attribute14,Attribute15,Attribute16,Attribute17,Attribute18,Attribute19,Attribute20
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,4,A121,67,A143,A152,2,A173,1,A192,A201
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,2,A121,22,A143,A152,1,A173,1,A191,A201
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,3,A121,49,A143,A152,1,A172,2,A191,A201
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,4,A122,45,A143,A153,1,A173,2,A191,A201
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,4,A124,53,A143,A153,2,A173,2,A191,A201


In [2]:
df_labels = pd.read_csv('../data/labels.csv')
df_labels.head(5)

,class
0,1
1,2
2,1
3,1
4,2


In [3]:
import json

with open("../data_metadata.json", "r", encoding="utf-8") as f:
    schema = json.load(f)

for col, element in schema.items():
    df.rename(columns={col: element['description']}, inplace=True)

In [4]:
df.head(5)

,Status of existing checking account,Duration,Credit history,Purpose,Credit amount,Savings account/bonds,Present employment since,Installment rate in percentage of disposable income,Personal status and sex,Other debtors / guarantors,Present residence since,Property,Age,Other installment plans,Housing,Number of existing credits at this bank,Job,Number of people being liable to provide maintenance for,Telephone,foreign worker
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,4,A121,67,A143,A152,2,A173,1,A192,A201
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,2,A121,22,A143,A152,1,A173,1,A191,A201
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,3,A121,49,A143,A152,1,A172,2,A191,A201
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,4,A122,45,A143,A153,1,A173,2,A191,A201
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,4,A124,53,A143,A153,2,A173,2,A191,A201


In [5]:
print(df.dtypes)

Status of existing checking account                           str
Duration                                                    int64
Credit history                                                str
Purpose                                                       str
Credit amount                                               int64
Savings account/bonds                                         str
Present employment since                                      str
Installment rate in percentage of disposable income         int64
Personal status and sex                                       str
Other debtors / guarantors                                    str
Present residence since                                     int64
Property                                                      str
Age                                                         int64
Other installment plans                                       str
Housing                                                       str
Number of 

In [6]:
df_labeled = pd.concat([df, df_labels], axis=1)
df_labeled.rename(columns={'class': 'Credit Risk'}, inplace=True)
df_labeled.head(5)

,Status of existing checking account,Duration,Credit history,Purpose,Credit amount,Savings account/bonds,Present employment since,Installment rate in percentage of disposable income,Personal status and sex,Other debtors / guarantors,...,Property,Age,Other installment plans,Housing,Number of existing credits at this bank,Job,Number of people being liable to provide maintenance for,Telephone,foreign worker,Credit Risk
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


Vérifier la corrélation entre la variable cible et les variables numériques

In [7]:
# correlation between numerical features and target variable
df_labeled_numeric = df_labeled.select_dtypes(include=['int64', 'float64'])
correlation_matrix = df_labeled_numeric.corr()
print(correlation_matrix['Credit Risk'].sort_values(ascending=False))

Credit Risk                                                 1.000000
Duration                                                    0.214927
Credit amount                                               0.154739
Installment rate in percentage of disposable income         0.072404
Present residence since                                     0.002967
Number of people being liable to provide maintenance for   -0.003015
Number of existing credits at this bank                    -0.045732
Age                                                        -0.091127
Name: Credit Risk, dtype: float64


tranformer les variables catégorielles en utilisant one-hot encoding puis regarder la corrélation avec la variable cible de nouveau

In [8]:
# Transorm Categorical features to numeric using one-hot encoding
df_labeled_encoded = pd.get_dummies(df_labeled, drop_first=True)
df_labeled_encoded.head(5)

,Duration,Credit amount,Installment rate in percentage of disposable income,Present residence since,Age,Number of existing credits at this bank,Number of people being liable to provide maintenance for,Credit Risk,Status of existing checking account_A12,Status of existing checking account_A13,...,Property_A124,Other installment plans_A142,Other installment plans_A143,Housing_A152,Housing_A153,Job_A172,Job_A173,Job_A174,Telephone_A192,foreign worker_A202
0,6,1169,4,4,67,2,1,1,False,False,...,False,False,True,True,False,False,True,False,True,False
1,48,5951,2,2,22,1,1,2,True,False,...,False,False,True,True,False,False,True,False,False,False
2,12,2096,2,3,49,1,2,1,False,False,...,False,False,True,True,False,True,False,False,False,False
3,42,7882,2,4,45,1,2,1,False,False,...,False,False,True,False,True,False,True,False,False,False
4,24,4870,3,4,53,2,2,2,False,False,...,True,False,True,False,True,False,True,False,False,False


In [9]:
# Correlation between features and target variable
correlation_matrix = df_labeled_encoded.corr()
print(correlation_matrix['Credit Risk'].sort_values(ascending=False))

Credit Risk                                                 1.000000
Duration                                                    0.214927
Credit amount                                               0.154739
Credit history_A31                                          0.134448
Property_A124                                               0.125750
Status of existing checking account_A12                     0.119581
Present employment since_A72                                0.106397
Housing_A153                                                0.081556
Personal status and sex_A92                                 0.075493
Installment rate in percentage of disposable income         0.072404
Purpose_A46                                                 0.070088
Other debtors / guarantors_A102                             0.062728
Other installment plans_A142                                0.050523
Credit history_A32                                          0.043722
Job_A174                          

### Entraînement des modèles

Arbre de décision

In [10]:
# training a decision tree classifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

X = df_labeled_encoded.drop('Credit Risk', axis=1)
y = df_labeled_encoded['Credit Risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           1       0.81      0.77      0.79       141
           2       0.51      0.56      0.53        59

    accuracy                           0.71       200
   macro avg       0.66      0.67      0.66       200
weighted avg       0.72      0.71      0.71       200

[[109  32]
 [ 26  33]]


Régression logistique

In [11]:
# training a simple logistic regression model
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

X = df_labeled_encoded.drop('Credit Risk', axis=1)
y = df_labeled_encoded['Credit Risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=10000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           1       0.83      0.90      0.86       141
           2       0.70      0.56      0.62        59

    accuracy                           0.80       200
   macro avg       0.77      0.73      0.74       200
weighted avg       0.79      0.80      0.79       200

[[127  14]
 [ 26  33]]


Forêt aléatoire

In [12]:
# training a random forest ensemble model
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

X = df_labeled_encoded.drop('Credit Risk', axis=1)
y = df_labeled_encoded['Credit Risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           1       0.77      0.91      0.83       141
           2       0.62      0.34      0.44        59

    accuracy                           0.74       200
   macro avg       0.70      0.63      0.64       200
weighted avg       0.73      0.74      0.72       200

[[129  12]
 [ 39  20]]
